# MotorPulse feature engineering — learn from the supplied dataset

This notebook uses one main data source: `MotorPulsePoc/motorpulse_mixed_motor_synthetic_v1.csv`. It walks through the file in small steps and explains what each step means without assuming machine-learning knowledge.

> **Important:** this CSV is synthetic (computer-generated), not sensor data from a real motor. It contains feature values already created by a script. It does not contain raw accelerometer samples. This notebook can explore and prepare those existing features, but it cannot recalculate vibration features from raw sensor readings using this CSV.

## What you will learn

1. Load and inspect the supplied mixed-motor CSV.
2. Tell raw sensor readings apart from pre-calculated features and labels.
3. Check data quality and select model input columns without accidentally using the answer.
4. Create one additional example feature from values already in the file.
5. Compare a motor with its own synthetic healthy baseline.
6. Understand what extra data is needed before using real sensor measurements or claiming fault detection.

## Three simple terms

| Term | Meaning | Example in this file |
|---|---|---|
| **Raw data** | The original readings directly recorded by a sensor | Each timestamped X/Y/Z acceleration sample. This CSV does **not** contain it. |
| **Feature** | A number calculated from measurements to summarize them | `accel_vector_rms_m_s2`, already present in this CSV. |
| **Target / label** | The answer a supervised model is asked to predict | `scenario_label`; it identifies a generated scenario, not a confirmed real fault. |

Think of a feature like a summary written from many readings. The model uses summaries as inputs (`X`) and, in supervised learning, tries to predict a separate answer (`y`).

## Step 1 — Know where the data came from

The dataset was created by `generate_motorpulse_synthetic_dataset.py`. It contains five motor-family labels, three simulated assets in each family, and six generated scenarios. Every row has `data_source = synthetic_generator_v1`.

The scenario examples are intentionally balanced for software demonstrations. They do not represent real failure rates or verified motor faults. `fault_status` and `fault_type_confirmed` are `not_assessed_synthetic` in every row.

In [28]:
from pathlib import Path
import csv
from collections import Counter, defaultdict
import numpy as np

# Search upward so this works from the repository, MotorPulsePoc, or notebook folder.
dataset_name = "motorpulse_mixed_motor_synthetic_v1.csv"
search_roots = [Path.cwd(), *Path.cwd().parents]
candidate_paths = []
for root in search_roots:
    candidate_paths.extend([root / dataset_name, root / "MotorPulsePoc" / dataset_name])

dataset_path = next((path for path in candidate_paths if path.is_file()), None)
if dataset_path is None:
    raise FileNotFoundError(
        f"Could not find {dataset_name}. Keep it in MotorPulsePoc or run this notebook from the project."
    )

with dataset_path.open(newline="", encoding="utf-8") as csv_file:
    reader = csv.DictReader(csv_file)
    rows = list(reader)
    columns = reader.fieldnames

print("File:", dataset_path)
print("Number of rows:", len(rows))
print("Number of columns:", len(columns))
print("Source values:", Counter(row["data_source"] for row in rows))

File: d:\GTA\ai-project\MotorPulsePoc\motorpulse_mixed_motor_synthetic_v1.csv
Number of rows: 1800
Number of columns: 39
Source values: Counter({'synthetic_generator_v1': 1800})


## Step 2 — Preview rows and identify motor/scenario groups

Each row is one **feature window** for one simulated asset. `asset_id` tells us which simulated motor it belongs to. `scenario_label` tells us which generator rule created that row. The scenario label is not a diagnosis from an inspection.

In [29]:
preview_columns = [
    "window_id", "asset_id", "motor_type", "drive_type",
    "scenario_label", "fault_status", "quality_status",
    "accel_vector_rms_m_s2", "temperature_c", "current_rms_a",
]
for row in rows[:5]:
    print({column: row[column] for column in preview_columns})

print("\nMotor families:", Counter(row["motor_type"] for row in rows))
print("Scenario labels:", Counter(row["scenario_label"] for row in rows))

{'window_id': 'SIM-01-01-001', 'asset_id': 'INDUCTION-01', 'motor_type': 'induction', 'drive_type': 'vfd', 'scenario_label': 'synthetic_healthy', 'fault_status': 'not_assessed_synthetic', 'quality_status': 'valid', 'accel_vector_rms_m_s2': '0.2685', 'temperature_c': '29.41', 'current_rms_a': '1.979'}
{'window_id': 'SIM-01-01-002', 'asset_id': 'INDUCTION-01', 'motor_type': 'induction', 'drive_type': 'vfd', 'scenario_label': 'synthetic_healthy', 'fault_status': 'not_assessed_synthetic', 'quality_status': 'valid', 'accel_vector_rms_m_s2': '0.2637', 'temperature_c': '31.08', 'current_rms_a': '2.266'}
{'window_id': 'SIM-01-01-003', 'asset_id': 'INDUCTION-01', 'motor_type': 'induction', 'drive_type': 'vfd', 'scenario_label': 'synthetic_healthy', 'fault_status': 'not_assessed_synthetic', 'quality_status': 'valid', 'accel_vector_rms_m_s2': '0.284', 'temperature_c': '28.6', 'current_rms_a': '2.077'}
{'window_id': 'SIM-01-01-004', 'asset_id': 'INDUCTION-01', 'motor_type': 'induction', 'drive_typ

## Step 3 — Which columns are features, context, or labels?

- **Existing numeric features:** vibration RMS/peak/frequency values, temperature values, and current RMS.
- **Context:** motor family, drive, driven equipment, RPM, load, sample rate, and window duration. These describe what was being measured and how the motor was operating.
- **Quality/provenance:** quality status, sensor profile, source, and mounting location.
- **Labels:** `scenario_label`, `fault_status`, and `fault_type_confirmed`.
- **Identifiers:** window, session, and asset IDs; useful for tracing and grouped testing, usually not model inputs.

Do not give a model the label it is supposed to predict. Doing that is like giving a student the answer key during an exam.

In [30]:
candidate_feature_columns = [
    "accel_x_rms_m_s2",
    "accel_y_rms_m_s2",
    "accel_z_rms_m_s2",
    "accel_vector_rms_m_s2",
    "accel_peak_to_peak_max_m_s2",
    "accel_crest_factor_max",
    "dominant_frequency_hz",
    "temperature_c",
    "temperature_rise_c",
    "current_rms_a",
]
context_columns = ["motor_type", "drive_type", "driven_equipment", "actual_rpm", "load_pct", "run_state"]
label_columns = ["scenario_label", "fault_status", "fault_type_confirmed"]
identifier_columns = ["window_id", "session_id", "asset_id", "recorded_at_utc", "split_group_asset_id"]

print("Candidate calculated features:", candidate_feature_columns)
print("Context (not raw vibration features):", context_columns)
print("Labels (never include the chosen target in X):", label_columns)
print("Identifiers (keep for traceability/group splits):", identifier_columns)

Candidate calculated features: ['accel_x_rms_m_s2', 'accel_y_rms_m_s2', 'accel_z_rms_m_s2', 'accel_vector_rms_m_s2', 'accel_peak_to_peak_max_m_s2', 'accel_crest_factor_max', 'dominant_frequency_hz', 'temperature_c', 'temperature_rise_c', 'current_rms_a']
Context (not raw vibration features): ['motor_type', 'drive_type', 'driven_equipment', 'actual_rpm', 'load_pct', 'run_state']
Labels (never include the chosen target in X): ['scenario_label', 'fault_status', 'fault_type_confirmed']
Identifiers (keep for traceability/group splits): ['window_id', 'session_id', 'asset_id', 'recorded_at_utc', 'split_group_asset_id']


## Step 4 — Check quality before analysis

Some rows intentionally represent a sensor-quality problem. Their numeric measurements are blank. A blank means “not available,” not zero. We exclude invalid rows from this example feature table, but keep them in storage so a separate sensor-health workflow can use them.

In [31]:
valid_rows = [
    row for row in rows
    if row["quality_status"] == "valid"
    and all(row[column].strip() != "" for column in candidate_feature_columns)
]
invalid_rows = [row for row in rows if row["quality_status"] != "valid"]

print("All rows:", len(rows))
print("Valid rows for this feature example:", len(valid_rows))
print("Invalid-quality rows kept aside:", len(invalid_rows))
print("Quality statuses:", Counter(row["quality_status"] for row in rows))

All rows: 1800
Valid rows for this feature example: 1500
Invalid-quality rows kept aside: 300
Quality statuses: Counter({'valid': 1500, 'synthetic_invalid_sensor_data': 300})


### Save the cleaned feature table for the next notebook

The Random Forest notebook is separate, so it cannot see this notebook's in-memory `valid_rows` variable. Save these cleaned rows to a CSV file. The model notebook will load this exact file; it will not recalculate features or repeat the cleaning here.

In [ ]:
cleaned_dataset_path = dataset_path.with_name("motorpulse_mixed_motor_cleaned_synthetic_v1.csv")
with cleaned_dataset_path.open("w", newline="", encoding="utf-8") as cleaned_file:
    writer = csv.DictWriter(cleaned_file, fieldnames=list(rows[0].keys()))
    writer.writeheader()
    writer.writerows(valid_rows)

print("Saved cleaned feature rows:", len(valid_rows))
print("Saved to:", cleaned_dataset_path)
print("Source remains synthetic; cleaning does not make it real sensor data.")

## Step 5 — Choose model inputs (`X`) and target (`y`)

For a practice classification exercise, use the 10 numeric feature columns as `X` and `scenario_label` as `y`. This only teaches a model to recognize the generator's scenarios. It does **not** teach real motor fault diagnosis.

We exclude `fault_status` because it is `not_assessed_synthetic` for every row. We also exclude IDs, timestamps, source labels, and the target itself from `X`. Context such as `motor_type` may be added later for a suitable model, but must be encoded and evaluated carefully.

In [32]:
X = np.array(
    [[float(row[column]) for column in candidate_feature_columns] for row in valid_rows],
    dtype=float,
)
y = [row["scenario_label"] for row in valid_rows]

print("X shape (windows, features):", X.shape)
print("y length (one target per window):", len(y))
print("Feature names:", candidate_feature_columns)
print("First row X:", X[0])
print("First row y:", y[0])
print("Fault status for first row:", valid_rows[0]["fault_status"])

X shape (windows, features): (1500, 10)
y length (one target per window): 1500
Feature names: ['accel_x_rms_m_s2', 'accel_y_rms_m_s2', 'accel_z_rms_m_s2', 'accel_vector_rms_m_s2', 'accel_peak_to_peak_max_m_s2', 'accel_crest_factor_max', 'dominant_frequency_hz', 'temperature_c', 'temperature_rise_c', 'current_rms_a']
First row X: [ 0.125   0.1416  0.1908  0.2685  0.8135  2.3711 24.643  29.41    6.74
  1.979 ]
First row y: synthetic_healthy
Fault status for first row: not_assessed_synthetic


## Step 6 — What these feature columns mean

The columns below are already present in the CSV. We are **reading** them, not calculating them from sensor samples in this notebook.

| Feature | Simple meaning | How real data would be calculated/captured |
|---|---|---|
| `accel_x_rms_m_s2`, `accel_y_rms_m_s2`, `accel_z_rms_m_s2` | Average vibration size on each direction | Calculate RMS from calibrated raw acceleration samples for that axis and window. |
| `accel_vector_rms_m_s2` | Overall vibration magnitude across all three axes | For synchronized centered samples: square each axis, add per sample, average, then square-root. |
| `accel_peak_to_peak_max_m_s2` | Largest swing among the axes | For each axis: maximum minus minimum; keep the largest axis range. |
| `accel_crest_factor_max` | Largest peak relative to RMS | Per axis: absolute peak divided by axis RMS; keep the maximum. |
| `dominant_frequency_hz` | Strongest repeating frequency found in the window | Calculate an FFT/PSD on adequately sampled raw vibration; this CSV's values are generated examples. |
| `temperature_c` | Sensor temperature, such as motor-casing temperature | Read from the temperature sensor with its own timestamp. Surface temperature is not winding temperature. |
| `temperature_rise_c` | Temperature above ambient | Requires both casing and ambient measurements aligned in time: casing minus ambient. |
| `current_rms_a` | RMS current summary | Capture from a calibrated current sensor or approved drive telemetry. Detailed electrical spectra need raw fast current samples. |

For actual raw-signal formulas, sampling limits, and calibration details, see the project's beginner guide and data dictionary.

## Step 7 — Understand where real IoT values will come from

This table describes the future real-device data path. A sensor or drive supplies measurements; software calculates summary features. The sample CSV is synthetic, so its `actual_rpm` value was made by the generator and did **not** come from an RPM sensor.

| Value | Real source in the IoT system | Sensor reading or software calculation? |
|---|---|---|
| `acceleration_samples` (X/Y/Z) | Triaxial accelerometer attached to the motor | Raw sensor samples |
| `accel_*_rms`, vector RMS, peak-to-peak, crest factor, frequency | The X/Y/Z acceleration samples plus sample rate | Calculated by edge/server feature code |
| `temperature_c` | DS18B20 attached at a documented casing location | Direct temperature measurement, converted/calibrated to °C |
| `ambient_temperature_c` | A separate ambient-air temperature sensor | Direct measurement; not currently installed in this prototype |
| `temperature_rise_c` | Casing and ambient temperature readings aligned by time | Calculated as casing °C minus ambient °C |
| `current_rms_a` | ACS712/approved current sensor, measured and calibrated by ESP32 | Current RMS is calculated by firmware from ADC samples; current prototype sends a one-second RMS summary |
| `actual_rpm` | Tachometer, encoder, or actual-speed feedback from a drive/VFD | Direct measured/drive telemetry; **not measured by the current ESP32** |
| `motor_type`, `drive_type` | Verified nameplate/asset setup or approved drive configuration | Asset metadata entered/confirmed by a person |

A motor rotates at `RPM / 60` revolutions per second. Dividing a vibration frequency by this running frequency gives an approximate **order** (how many times the frequency occurs per rotation). In a real deployment RPM must be measured and timestamp-aligned with the vibration window. If RPM is unavailable, leave the order unavailable; do not invent an RPM value.

In [33]:
# The CSV field is named actual_rpm, but in this file the value is SIMULATED.
# In the real IoT pipeline this must be replaced with a timestamped tachometer,
# encoder, or trusted VFD actual-speed reading. The ESP32 currently has no RPM sensor.
for row in valid_rows:
    simulated_rpm = float(row["actual_rpm"])
    running_frequency_hz = simulated_rpm / 60.0
    row["synthetic_dominant_frequency_order"] = float(row["dominant_frequency_hz"]) / running_frequency_hz

print("First row CSV RPM (synthetic, not sensor-measured):", valid_rows[0]["actual_rpm"])
print("First row dominant frequency (Hz):", valid_rows[0]["dominant_frequency_hz"])
print("Calculated synthetic approximate order:", round(valid_rows[0]["synthetic_dominant_frequency_order"], 3))
print("This synthetic example exists only in notebook memory; the original CSV was not changed.")

First row CSV RPM (synthetic, not sensor-measured): 1448.7
First row dominant frequency (Hz): 24.643
Calculated synthetic approximate order: 1.021
This synthetic example exists only in notebook memory; the original CSV was not changed.


## Step 8 — Compare one asset with its own healthy baseline

A **baseline** is a reference for how one specific motor normally behaves when it is known to be healthy. It helps answer: “Is this new measurement unusual for this same motor under a comparable operating condition?” It does not answer: “Which part is broken?”

Why use it? Motors differ, even when they have the same type. Sensor mounting, speed, load, and the driven machine affect the readings. A baseline for Motor A should not be used for Motor B.

This example selects generated healthy rows for `INDUCTION-01`, summarizes that motor's vector RMS, and compares one generated changed window. The CSV's scenario label is used only to choose a teaching example. A real-time system does not know the scenario label in advance; it scores each incoming window. Because this CSV is synthetic, the result demonstrates arithmetic only—not real fault detection.

In [34]:
asset_to_demo = "INDUCTION-01"
healthy_rows = [
    row for row in valid_rows
    if row["asset_id"] == asset_to_demo and row["scenario_label"] == "synthetic_healthy"
]
changed_rows = [
    row for row in valid_rows
    if row["asset_id"] == asset_to_demo and row["scenario_label"] == "synthetic_vibration_deviation"
]

healthy_rms = np.array([float(row["accel_vector_rms_m_s2"]) for row in healthy_rows])
observed_rms = float(changed_rows[0]["accel_vector_rms_m_s2"])
baseline_median = float(np.median(healthy_rms))
baseline_mad = float(np.median(np.abs(healthy_rms - baseline_median)))
scale_floor = max(abs(baseline_median) * 0.10, 0.05)
robust_scale = max(1.4826 * baseline_mad, scale_floor)
distance_from_baseline = abs(observed_rms - baseline_median) / robust_scale

print("Asset:", asset_to_demo)
print("Synthetic healthy windows:", len(healthy_rows))
print(f"Healthy median vector RMS: {baseline_median:.3f} m/s²")
print(f"Example changed window RMS: {observed_rms:.3f} m/s²")
print(f"Robust distance from this example baseline: {distance_from_baseline:.2f}")
print("Meaning: this generated value differs from generated healthy values; it does not prove a real fault.")

Asset: INDUCTION-01
Synthetic healthy windows: 20
Healthy median vector RMS: 0.269 m/s²
Example changed window RMS: 0.558 m/s²
Robust distance from this example baseline: 5.79
Meaning: this generated value differs from generated healthy values; it does not prove a real fault.


### Read the baseline code in plain language

1. `asset_to_demo` chooses the motor whose baseline we want. In production this would be the `asset_id` from the incoming IoT message.
2. `healthy_rows` picks windows for that same motor which a person has marked healthy. In this synthetic exercise, `synthetic_healthy` is only a generated label, not a human inspection.
3. `changed_rows` picks one generated example to compare. A real incoming window is not selected by a known label; it arrives from the sensor and is analyzed.
4. `healthy_rms` collects the motor's healthy vector-RMS values. The **median** is the middle value after sorting and is used as a typical reference.
5. `baseline_mad` is the median distance from that middle value. It describes normal spread while being less affected by extreme values than a regular average.
6. `robust_scale` converts that spread to a usable scale. The floor prevents division by zero or a tiny number when the generated healthy values are nearly identical.
7. `distance_from_baseline` measures how many robust scale units the new value is from the baseline median. A larger number means “more unusual compared with this reference.” It is **not** a probability of failure.

The current API applies a similar median/MAD comparison to five values (three calculated vibration features plus temperature and current), then reports the largest deviation. Its prototype thresholds are warning at 3 and critical at 6 robust deviations; these thresholds need validation on real, later data before operational use.

### How this should work for a real motor in real time

1. **Commission the asset:** a technician confirms the motor ID, sensor positions, and operating conditions. Capture several sessions while the motor is inspected and known healthy.
2. **Build and save its baseline:** calculate the same features on valid healthy windows for that asset and each supported speed/load regime. Save the baseline values, feature-schema version, sensor setup, and date.
3. **Receive a new IoT window:** the device sends timestamped sensor readings. The collector checks gaps, units, stale slow-sensor readings, and operating state. Invalid data should produce a data-quality warning, not a health score.
4. **Calculate the same features:** use the identical feature formulas, units, window length, sample rate, and preprocessing used during baseline creation.
5. **Select the matching baseline:** use the incoming asset ID and a measured operating regime. If there is no suitable baseline for that speed/load or the configuration changed, return `baseline unavailable/out of coverage` rather than compare against a mismatched motor.
6. **Score the window:** compare each feature with that motor's stored healthy median/MAD. The output says which feature differs and by how much. It is an anomaly flag, not a confirmed fault.
7. **Show and review:** the dashboard shows time, motor, feature value, healthy reference, data quality, and operating context. A technician acknowledges and inspects it; inspection results become labels later.
8. **Control alert noise:** a production service should aggregate repeated windows and use validated persistence/hysteresis rules. The current prototype scores one API request at a time; it does not yet implement a complete alert lifecycle or automatically connect MQTT to the AI API/dashboard.

Never automatically add every new reading to the healthy baseline. If a motor is slowly degrading, continuously learning from it could teach the system that the degradation is normal. Baseline updates should be reviewed and versioned.

### Why separate baselines matter

Healthy values vary with the motor, sensor mounting, drive, speed, and load. A family name such as `induction` is not enough to define one healthy vibration number for every induction motor. Start with one baseline per physical motor and comparable operating range. For variable speed, record measured RPM/load and either collect useful regimes or use a validated condition-aware method.

In [35]:
healthy_assets = defaultdict(list)
for row in valid_rows:
    if row["scenario_label"] == "synthetic_healthy":
        healthy_assets[row["asset_id"]].append(float(row["accel_vector_rms_m_s2"]))

print(f"{'ASSET':18s} {'TYPE':12s} {'HEALTHY MEDIAN RMS (m/s²)':>28s}")
print("-" * 62)
for asset_id in sorted(healthy_assets):
    asset_rows = [row for row in valid_rows if row["asset_id"] == asset_id]
    motor_type = asset_rows[0]["motor_type"]
    median_rms = float(np.median(healthy_assets[asset_id]))
    print(f"{asset_id:18s} {motor_type:12s} {median_rms:>28.3f}")

print("\nThese medians are generator output, not measured health limits.")

ASSET              TYPE            HEALTHY MEDIAN RMS (m/s²)
--------------------------------------------------------------
BLDC-07            bldc                                0.165
BLDC-08            bldc                                0.185
BLDC-09            bldc                                0.146
BRUSHED_DC-10      brushed_dc                          0.240
BRUSHED_DC-11      brushed_dc                          0.301
BRUSHED_DC-12      brushed_dc                          0.205
INDUCTION-01       induction                           0.269
INDUCTION-02       induction                           0.328
INDUCTION-03       induction                           0.255
PMSM-04            pmsm                                0.194
PMSM-05            pmsm                                0.229
PMSM-06            pmsm                                0.304
STEPPER-13         stepper                             0.334
STEPPER-14         stepper                             0.358
STEPPER-15         ste

## Step 9 — What is the target, and how do we know fault vs. no fault?

In this synthetic file, `scenario_label` can be used as `y` only to practice predicting the generator's scenarios. The labels include vibration/thermal/current deviations and operating/sensor scenarios. A deviation is not automatically a mechanical fault.

For **real** fault classification, use a separate `fault_status` label based on inspection evidence:

- `confirmed_fault`: a qualified inspection found a fault and linked it to that asset/time period.
- `confirmed_no_fault`: inspection reviewed that period and confirmed no fault.
- `unknown`: not inspected or uncertain; do not train it as a negative example.
- `sensor_issue` and `operating_change`: keep separate from motor fault labels.

No alert does not prove no fault, and an alert does not prove a fault. In this CSV, every row's fault status is `not_assessed_synthetic`.

## Step 10 — Keep train and test data separated by asset

If windows from the same simulated motor appear in both training and testing, the test may be too easy because the rows share the same generator profile. A better demonstration holds out entire assets. For a real evaluation, hold out complete motors and/or later sessions, then report results per motor family and operating condition.

The split below only demonstrates grouping. It does not train a model or prove accuracy.

In [36]:
assets_by_type = defaultdict(list)
for row in valid_rows:
    if row["asset_id"] not in assets_by_type[row["motor_type"]]:
        assets_by_type[row["motor_type"]].append(row["asset_id"])

test_assets = {sorted(asset_ids)[-1] for asset_ids in assets_by_type.values()}
train_rows = [row for row in valid_rows if row["asset_id"] not in test_assets]
test_rows = [row for row in valid_rows if row["asset_id"] in test_assets]

print("Held-out test assets:", sorted(test_assets))
print("Training rows:", len(train_rows), "from", len({row['asset_id'] for row in train_rows}), "assets")
print("Test rows:", len(test_rows), "from", len({row['asset_id'] for row in test_rows}), "assets")
print("Test split contains these motor types:", sorted({row['motor_type'] for row in test_rows}))

Held-out test assets: ['BLDC-09', 'BRUSHED_DC-12', 'INDUCTION-03', 'PMSM-06', 'STEPPER-15']
Training rows: 1000 from 10 assets
Test rows: 500 from 5 assets
Test split contains these motor types: ['bldc', 'brushed_dc', 'induction', 'pmsm', 'stepper']


## What the current MotorPulse API uses

The current `backend/ai-service/main.py` accepts raw X/Y/Z acceleration samples, sample rate, temperature, and current. It calculates three vibration features:

1. `acceleration_vector_rms_m_s2`
2. `acceleration_peak_to_peak_m_s2` (largest of the three axis ranges)
3. `dominant_frequency_hz` (one strongest non-zero FFT bin)

It uses `temperature_c` and `current_amps` directly as the other two values, then compares them with a per-motor median/MAD baseline. The request accepts optional RPM, but the current feature calculation ignores it. The API reports a baseline deviation—not a confirmed fault or failure prediction.

The CSV has extra candidate features for learning about data preparation; they are not all implemented in the current API.

## What is needed to calculate features from real sensors?

This CSV cannot be used to recompute vibration features because it does not contain raw vibration samples. For real feature engineering, capture rows such as:

| timestamp | asset_id | accel_x_m_s2 | accel_y_m_s2 | accel_z_m_s2 |
|---|---|---:|---:|---:|
| device sample time | physical motor ID | measured X | measured Y | measured Z |

Keep temperature, current, RPM, and load with their own timestamps and measurement intervals. Then quality-check the readings, group vibration samples into windows, apply documented preprocessing, and calculate the features above. The existing MQTT logger can store raw readings, but its source must be set to `real` only when connected to physical sensors; its default source is simulated.

Use the blank real-capture template and the dataset dictionary for column requirements. Do not call synthetic data “real,” and do not deliberately damage a working motor to obtain fault examples.

## Summary checklist

- [ ] I know whether the data is real, test-rig, public, or synthetic.
- [ ] I know whether each column is a raw measurement, calculated feature, context value, ID, or label.
- [ ] I excluded invalid sensor rows and did not replace missing readings with zero.
- [ ] I kept target labels out of the model inputs.
- [ ] I used a separate baseline for each asset/operating regime.
- [ ] I did not treat a synthetic scenario or anomaly as a confirmed fault.
- [ ] I split model evaluation by whole assets/sessions instead of neighboring rows.

## Glossary

- **Window:** a short group of nearby sensor samples.
- **Feature:** a calculated summary of a window.
- **Baseline:** a verified reference for one motor under stated conditions.
- **Anomaly:** a change from the expected baseline; not necessarily a fault.
- **RMS:** a summary of signal magnitude, calculated using squares, an average, and a square root.
- **MAD:** median absolute deviation; a robust way to describe variation in baseline values.
- **Target (`y`):** the answer a supervised model is asked to predict.

For detailed column definitions, see `MOTORPULSE_DATASET_DICTIONARY.md` in the MotorPulsePoc folder.